# Semi Structured JSONPath And XML Mapping

This notebook exercises semi-structured data handling: real JSONPath selection with exact path evidence, array-to-edge construction, dotted-path fallback, and XML flattening.

In [ ]:
from pathlib import Path
import sys, json, tempfile

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next((p for p in Path.cwd().parents if (p / 'src').exists()), Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print(f'Repo root: {ROOT}')

In [ ]:
from kg_construction.domain.models import SourceRecord
from kg_construction.infrastructure.mapping import (
    DottedPathJsonMapper, JsonArrayEdgeMapping, JsonPathMapper, JsonPathMappingConfig, JsonPropertyMapping,
)
from kg_construction.infrastructure.loaders import XMLLoader, APIResponseLoader
from kg_construction.infrastructure.graph_builders import PropertyGraphBuilder
from kg_construction.infrastructure.ontology.validation import EvidenceRequiredValidator

payload = {
    'service': {'id': 'svc-rec-api', 'name': 'Recommendation API'},
    'errors': [
        {'code': 'E1042', 'message': 'Timeout'},
        {'code': 'E1043', 'message': 'Retry limit exceeded'},
    ],
}
source = SourceRecord.from_json(payload, source_name='memory://jsonpath-demo')
mapper = JsonPathMapper(JsonPathMappingConfig(
    node_type='ServicePayload',
    name_path='$.service.name',
    property_paths=[
        JsonPropertyMapping(property_name='service_name', path='$.service.name'),
        JsonPropertyMapping(property_name='first_error_code', path='$.errors[0].code'),
    ],
    array_edges=[JsonArrayEdgeMapping(
        path='$.errors[*]', relationship_type='HAS_ERROR', target_node_type='Error', target_name_path='$.code'
    )],
))
batch = mapper.map_source(source)
print({'entities': len(batch.entities), 'relations': len(batch.relations)})
for entity in batch.entities:
    print(entity.entity_type, entity.name, entity.properties.get('_json_path'), entity.properties.get('_json_property_paths'))
for relation in batch.relations:
    print(relation.relation_type, relation.properties['source_path'])

In [ ]:
graph = PropertyGraphBuilder().build(batch, validation_report=EvidenceRequiredValidator().validate(batch))
print({'nodes': len(graph.nodes), 'edges': len(graph.edges)})
print(graph.edges[0].properties['source_paths'])

In [ ]:
# DottedPathJsonMapper is the lightweight flattener. It is useful when full JSONPath rules are unnecessary.
dotted_batch = DottedPathJsonMapper(node_type='ServiceRecord').map_source(source)
print(dotted_batch.entities[0].properties)

In [ ]:
fixtures = ROOT / 'tests' / 'fixtures' / 'enterprise_loaders'
xml_source = XMLLoader().load(fixtures / 'services.xml')[0]
api_source = APIResponseLoader().load(fixtures / 'api_response.json')[0]
print('XML rows:', xml_source.records)
print('XML evidence:', xml_source.metadata['source_evidence'][0]['location'])
print('API rows:', api_source.records)